# Amazon ML Challenge - Model Training and Evaluation

This notebook implements a step-by-step approach to train and evaluate models for the Amazon ML price prediction challenge.

**Evaluation Metric**: SMAPE (Symmetric Mean Absolute Percentage Error)

**Goal**: Predict product prices based on catalog content and images

## Step 1: Import Required Libraries and Setup

In [1]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
import os
import time

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")
print(f"Working directory: {os.getcwd()}")

✅ Libraries imported successfully!
Working directory: e:\IITJ\Amazon_ML\student_resource\src


## Step 2: Define SMAPE Evaluation Metric

SMAPE (Symmetric Mean Absolute Percentage Error) is our evaluation metric for this challenge.

In [2]:
def calculate_smape(y_true, y_pred):
    """
    Calculate SMAPE (Symmetric Mean Absolute Percentage Error)
    
    SMAPE = (100 / n) * Σ(|y_true - y_pred| / ((|y_true| + |y_pred|) / 2))
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
        
    Returns:
        SMAPE score (lower is better)
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Handle edge cases
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    
    # Avoid division by zero
    mask = denominator != 0
    smape = np.zeros_like(y_true, dtype=float)
    smape[mask] = np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]
    
    return 100 * np.mean(smape)

# Test the SMAPE function
test_true = [100, 200, 300]
test_pred = [110, 190, 310]
test_smape = calculate_smape(test_true, test_pred)

print(f"✅ SMAPE function defined successfully!")
print(f"Test SMAPE score: {test_smape:.4f}")
print("Lower SMAPE is better (perfect score = 0)")

✅ SMAPE function defined successfully!
Test SMAPE score: 5.9769
Lower SMAPE is better (perfect score = 0)


## Step 3: Load and Explore the Dataset

We'll load the training data and explore its structure.

In [3]:
# Define file paths
DATASET_PATH = '../dataset/'
TRAIN_FILE = 'train.csv'
TEST_FILE = 'test.csv'

print("📁 Loading dataset...")
print(f"Dataset path: {DATASET_PATH}")

# Check if files exist
train_path = os.path.join(DATASET_PATH, TRAIN_FILE)
test_path = os.path.join(DATASET_PATH, TEST_FILE)

if os.path.exists(train_path):
    print(f"✅ Training file found: {train_path}")
else:
    print(f"❌ Training file not found: {train_path}")

if os.path.exists(test_path):
    print(f"✅ Test file found: {test_path}")
else:
    print(f"❌ Test file not found: {test_path}")

# Load a small sample first to understand the structure
print("\n📊 Loading a sample of training data...")

📁 Loading dataset...
Dataset path: ../dataset/
✅ Training file found: ../dataset/train.csv
✅ Test file found: ../dataset/test.csv

📊 Loading a sample of training data...


In [4]:
# Load a sample of training data to understand structure
try:
    # Load first 1000 rows to understand the data structure
    train_sample = pd.read_csv(train_path, nrows=1000)
    
    print(f"✅ Loaded sample with {len(train_sample)} rows")
    print(f"📏 Dataset shape: {train_sample.shape}")
    print(f"📋 Columns: {list(train_sample.columns)}")
    
    # Display basic info
    print("\n📊 Dataset Info:")
    print(train_sample.info())
    
    print("\n📈 Price Statistics:")
    print(train_sample['price'].describe())
    
    print("\n🔍 Sample Data:")
    print(train_sample.head(3))
    
except Exception as e:
    print(f"❌ Error loading data: {e}")

✅ Loaded sample with 1000 rows
📏 Dataset shape: (1000, 4)
📋 Columns: ['sample_id', 'catalog_content', 'image_link', 'price']

📊 Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sample_id        1000 non-null   int64  
 1   catalog_content  1000 non-null   object 
 2   image_link       1000 non-null   object 
 3   price            1000 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 31.4+ KB
None

📈 Price Statistics:
count    1000.000000
mean       24.756900
std        31.648316
min         0.740000
25%         6.852500
50%        14.950000
75%        30.620000
max       337.670000
Name: price, dtype: float64

🔍 Sample Data:
   sample_id                                    catalog_content  \
0      33127  Item Name: La Victoria Green Taco Sauce Mild, ...   
1     198967  Item Name: Salerno Cookies, 

## Step 4: Check for Pre-processed Features

Let's check if we have the enhanced datasets with text features from EDA analysis.

In [6]:
# Check for enhanced datasets from EDA analysis
enhanced_files = [
    'train_with_text_features.csv',
    'train_text_top96_features.csv',
    'train_with_top96_text_features_combined.csv'
]

print("🔍 Checking for enhanced datasets:")
available_files = []

for file in enhanced_files:
    file_path = os.path.join(DATASET_PATH, file)
    if os.path.exists(file_path):
        print(f"✅ Found: {file}")
        available_files.append(file)
    else:
        print(f"❌ Not found: {file}")

if available_files:
    print(f"\n📊 We have {len(available_files)} enhanced dataset(s) available!")
    
    # Use the most comprehensive one
    if 'train_with_top96_text_features_combined.csv' in available_files:
        selected_file = 'train_with_top96_text_features_combined.csv'
        print(f"🎯 Using: {selected_file} (most comprehensive)")
    else:
        selected_file = available_files[0]
        print(f"🎯 Using: {selected_file}")
        
else:
    print("⚠️  No enhanced datasets found. We'll work with the original data.")
    selected_file = TRAIN_FILE

🔍 Checking for enhanced datasets:
✅ Found: train_with_text_features.csv
✅ Found: train_text_top96_features.csv
✅ Found: train_with_top96_text_features_combined.csv

📊 We have 3 enhanced dataset(s) available!
🎯 Using: train_with_top96_text_features_combined.csv (most comprehensive)


## Step 5: Load Enhanced Dataset

Loading the enhanced dataset with text features for training.

In [7]:
# Load the enhanced dataset
enhanced_path = os.path.join(DATASET_PATH, selected_file)

print(f"📚 Loading enhanced dataset: {selected_file}")
print("⏳ This might take a moment for large datasets...")

try:
    # Load a sample first to check structure
    df_sample = pd.read_csv(enhanced_path, nrows=5)
    
    print(f"✅ Sample loaded successfully!")
    print(f"📏 Dataset has {len(df_sample.columns)} columns")
    print(f"📋 Column types:")
    
    # Identify different types of columns
    original_cols = ['sample_id', 'catalog_content', 'image_link', 'price']
    text_feature_cols = [col for col in df_sample.columns if col not in original_cols]
    
    print(f"   - Original columns: {len(original_cols)}")
    print(f"   - Text feature columns: {len(text_feature_cols)}")
    
    print(f"\n📊 Text features preview (first 10):")
    print(text_feature_cols[:10])
    
except Exception as e:
    print(f"❌ Error loading enhanced dataset: {e}")
    print("📝 Will proceed with original dataset")

📚 Loading enhanced dataset: train_with_top96_text_features_combined.csv
⏳ This might take a moment for large datasets...
✅ Sample loaded successfully!
📏 Dataset has 100 columns
📋 Column types:
   - Original columns: 4
   - Text feature columns: 96

📊 Text features preview (first 10):
['has_flavor', 'has_tea', 'has_ounce', 'has_natural', 'has_product', 'has_free', 'has_organic', 'has_pack', 'has_coffee', 'has_taste']


## Step 6: Load Training Data (Subset for Initial Training)

Let's load a subset of the data for initial model training and validation.

In [8]:
# Load a subset for initial training (adjust sample_size as needed)
SAMPLE_SIZE = 10000  # Start with 10k samples for faster training

print(f"📊 Loading {SAMPLE_SIZE:,} samples for training...")

# Load the data
df = pd.read_csv(enhanced_path, nrows=SAMPLE_SIZE)

print(f"✅ Loaded {len(df):,} samples")
print(f"📏 Dataset shape: {df.shape}")

# Separate features and target
original_cols = ['sample_id', 'catalog_content', 'image_link', 'price']
feature_cols = [col for col in df.columns if col not in original_cols]

# Extract features and target
X = df[feature_cols].copy()
y = df['price'].copy()

print(f"\n🎯 Training setup:")
print(f"   - Features (X): {X.shape}")
print(f"   - Target (y): {y.shape}")
print(f"   - Feature columns: {len(feature_cols)}")

# Check for missing values
print(f"\n🔍 Data quality check:")
print(f"   - Missing values in features: {X.isnull().sum().sum()}")
print(f"   - Missing values in target: {y.isnull().sum()}")

# Basic statistics
print(f"\n📈 Target variable (price) statistics:")
print(y.describe())

📊 Loading 10,000 samples for training...
✅ Loaded 10,000 samples
📏 Dataset shape: (10000, 100)

🎯 Training setup:
   - Features (X): (10000, 96)
   - Target (y): (10000,)
   - Feature columns: 96

🔍 Data quality check:
   - Missing values in features: 10000
   - Missing values in target: 0

📈 Target variable (price) statistics:
count    10000.000000
mean        23.678313
std         31.692956
min          0.130000
25%          6.840000
50%         13.995000
75%         28.470000
max        613.580000
Name: price, dtype: float64
✅ Loaded 10,000 samples
📏 Dataset shape: (10000, 100)

🎯 Training setup:
   - Features (X): (10000, 96)
   - Target (y): (10000,)
   - Feature columns: 96

🔍 Data quality check:
   - Missing values in features: 10000
   - Missing values in target: 0

📈 Target variable (price) statistics:
count    10000.000000
mean        23.678313
std         31.692956
min          0.130000
25%          6.840000
50%         13.995000
75%         28.470000
max        613.580000
N

## Step 7: Data Preprocessing and Train-Test Split

Handle missing values and split the data for training and validation.

In [9]:
# Handle missing values
print("🔧 Preprocessing data...")

# Check missing value patterns
missing_info = X.isnull().sum()
missing_cols = missing_info[missing_info > 0]

if len(missing_cols) > 0:
    print(f"⚠️  Found missing values in {len(missing_cols)} columns")
    print("Top 5 columns with missing values:")
    print(missing_cols.head())
    
    # For text features, missing values likely mean the feature is absent (0)
    print("🔄 Filling missing values with 0 (feature absent)")
    X_clean = X.fillna(0)
else:
    print("✅ No missing values found")
    X_clean = X.copy()

# Verify no missing values remain
remaining_missing = X_clean.isnull().sum().sum()
print(f"✅ Missing values after cleaning: {remaining_missing}")

# Split the data
print(f"\n📊 Splitting data into train/validation sets...")
X_train, X_val, y_train, y_val = train_test_split(
    X_clean, y, 
    test_size=0.2, 
    random_state=42,
    stratify=None  # We'll use random split for regression
)

print(f"✅ Data split completed:")
print(f"   - Training set: {X_train.shape[0]:,} samples")
print(f"   - Validation set: {X_val.shape[0]:,} samples")
print(f"   - Features: {X_train.shape[1]}")

# Quick stats on train/val sets
print(f"\n📈 Price distribution:")
print(f"   - Train mean: ${y_train.mean():.2f}")
print(f"   - Validation mean: ${y_val.mean():.2f}")
print(f"   - Train std: ${y_train.std():.2f}")
print(f"   - Validation std: ${y_val.std():.2f}")

🔧 Preprocessing data...
⚠️  Found missing values in 1 columns
Top 5 columns with missing values:
pos_Food & Beverage    10000
dtype: int64
🔄 Filling missing values with 0 (feature absent)
✅ Missing values after cleaning: 0

📊 Splitting data into train/validation sets...
✅ Data split completed:
   - Training set: 8,000 samples
   - Validation set: 2,000 samples
   - Features: 96

📈 Price distribution:
   - Train mean: $23.67
   - Validation mean: $23.70
   - Train std: $31.70
   - Validation std: $31.67
